# DINO pretrain + k-NN / linear probe на CIFAR-10

DINO — Self-Distillation with No Labels (Caron et al., 2021, arXiv:2104.14294) на примитивах `spartan_torch`:

1. **Примитивы библиотеки.** `DINOProjectionHead`, `DINOLoss`, `Centering`, `Sharpening`, `MomentumEncoder`.
2. **Student + momentum-teacher.** Учитель — EMA-копия студента; momentum растёт по косинусу 0.996 → 1.0. Лосс — кросс-энтропия между распределениями студента и учителя (центр + sharpen, grad-stop на учителе).
3. **Multi-crop.** 2 глобальных + N локальных вью; глобальные идут в обоих, локальные — только в студента.
4. **Оценка.** Pretrain → k-NN по фичам учителя + linear probe.

Редуцированная конфигурация (CIFAR-10, ViT-Tiny-подобный) — быстрая проверка пайплайна.

Запуск: `uv sync --extra dev --extra experiments`, затем `uv run jupyter lab`.

In [ ]:
# === Setup: локально / devcontainer / Colab ===
import sys

IN_COLAB = "google.colab" in sys.modules
MLFLOW_ENABLED = not IN_COLAB

if IN_COLAB:
    import subprocess
    from pathlib import Path
    PROJECT_ROOT = Path("/content/spartan-torch")
    if not (PROJECT_ROOT / ".git").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/mievst/spartan-torch.git", str(PROJECT_ROOT)],
            check=True,
        )
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
    sys.path.insert(0, str(PROJECT_ROOT / "experiments"))
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U",
         "--upgrade-strategy", "eager", "numpy", "scipy"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e",
         f"{PROJECT_ROOT}\[experiments,dev]"],
        check=True,
    )
else:
    PROJECT_ROOT = None

print(f"IN_COLAB={IN_COLAB} | MLFLOW_ENABLED={MLFLOW_ENABLED} | python={sys.version.split()[0]}")

## 0. Константы и пути

In [ ]:
from pathlib import Path

import torch

ROOT = (PROJECT_ROOT / "experiments/vit/dino") if IN_COLAB else Path.cwd()
DATA_DIR = ROOT / "data"
CKPT_DIR = ROOT / "checkpoints"
DATA_DIR.mkdir(exist_ok=True)
CKPT_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"cwd: {ROOT} | device: {DEVICE}")

# --- training ---
EPOCHS = 60
BATCH_SIZE = 64
NUM_WORKERS = 0
SEED = 0

# --- DINO (ViT-Tiny-подобный backbone) ---
IMG_SIZE = 32
PATCH_SIZE = 4
EMBED_DIM = 192
DEPTH = 6
NUM_HEADS = 3
HEAD_HIDDEN = 512
HEAD_OUT = 4096
TEACHER_TEMP = 0.04
STUDENT_TEMP = 0.1
CENTER_MOMENTUM = 0.9
BASE_MOMENTUM = 0.996
LR = 5e-4
WEIGHT_DECAY = 0.04
WARMUP_EPOCHS = 5

# --- multi-crop ---
GLOBAL_CROPS = 2
LOCAL_CROPS = 4
LOCAL_IMG = 16
GLOBAL_SCALE = (0.4, 1.0)
LOCAL_SCALE = (0.08, 0.4)

torch.manual_seed(SEED)
torch.set_float32_matmul_precision("medium")

MLFLOW_TRACKING_URI = "http://localhost:5000"
MLFLOW_EXPERIMENT_NAME = "dino-cifar10" 

## 1. Модель

In [ ]:
import sys
sys.path.insert(0, str(ROOT))

from dino_model import DINOLightning

backbone = dict(
    img_size=IMG_SIZE, patch_size=PATCH_SIZE, in_channels=3,
    embed_dim=EMBED_DIM, depth=DEPTH, num_heads=NUM_HEADS, num_classes=10,
    ff_hidden_size=EMBED_DIM * 4,
)
dino = DINOLightning(
    backbone=backbone,
    head_hidden_dim=HEAD_HIDDEN, head_out_dim=HEAD_OUT,
    teacher_temp=TEACHER_TEMP, student_temp=STUDENT_TEMP,
    center_momentum=CENTER_MOMENTUM, momentum=BASE_MOMENTUM,
    lr=LR, weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS, max_epochs=EPOCHS,
)
n = sum(p.numel() for p in dino.student.parameters() if p.requires_grad)
print(f"DINO student params: {n:,}")

## 2. Данные

In [ ]:
from torchvision import datasets, transforms

MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)


def global_transform():
    return transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=GLOBAL_SCALE),
        transforms.RandomHorizontalFlip(),
        transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
        transforms.RandomGrayscale(p=0.2),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])


def local_crop_transform():
    return transforms.Compose([
        transforms.RandomResizedCrop(LOCAL_IMG, scale=LOCAL_SCALE),
        transforms.RandomHorizontalFlip(),
        transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])


plain_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])


class MultiViewCIFAR10(datasets.CIFAR10):
    # Возвращает [global x N, local x M] вью в виде списка тензоров.

    def __init__(self, *args, global_crops=2, local_crops=4, **kw):
        super().__init__(*args, transform=None, **kw)
        self.global_crops = global_crops
        self.local_crops = local_crops
        self.gtf = global_transform()
        self.ltf = local_crop_transform()

    def __getitem__(self, i):
        img, _ = super().__getitem__(i)
        views = [self.gtf(img) for _ in range(self.global_crops)]
        views += [self.ltf(img) for _ in range(self.local_crops)]
        return views


def dino_collate(batch):
    # batch: list of [список вью] → (student_views, teacher_views)
    views = [torch.stack([b[i] for b in batch]) for i in range(len(batch[0]))]
    return views, views[:2]  # student: global+local, teacher: global

In [ ]:
train_ds = MultiViewCIFAR10(
    DATA_DIR, train=True, download=True,
    global_crops=GLOBAL_CROPS, local_crops=LOCAL_CROPS,
)
val_dino = MultiViewCIFAR10(
    DATA_DIR, train=False, download=True,
    global_crops=GLOBAL_CROPS, local_crops=0,
)
train_plain = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=plain_transform)
val_ds = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=plain_transform)

train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=dino_collate,
)
val_dino_loader = torch.utils.data.DataLoader(
    val_dino, batch_size=BATCH_SIZE * 2, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=dino_collate,
)
train_plain_loader = torch.utils.data.DataLoader(
    train_plain, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
print(f"Train: {len(train_ds)} (x{GLOBAL_CROPS}+{LOCAL_CROPS} вью) | Val: {len(val_ds)}")

## 3. Pretrain

In [ ]:
from collections import defaultdict
from lightning.pytorch.callbacks import Callback, ModelCheckpoint


class MetricsCollector(Callback):
    def __init__(self):
        super().__init__()
        self.metrics = defaultdict(list)

    def on_validation_epoch_end(self, trainer, pl_module):
        for k, v in trainer.logged_metrics.items():
            self.metrics[k].append(v.cpu().item())


metrics_cb = MetricsCollector()
pretrain_cb = ModelCheckpoint(
    dirpath=CKPT_DIR, filename="dino-pretrain-{epoch}",
    monitor="val/proto_acc", mode="max", save_top_k=1, save_last=True,
)

In [ ]:
import socket
from urllib.parse import urlparse
from lightning.pytorch.loggers import MLFlowLogger


def _mlflow_reachable(uri, timeout=2.0):
    if not MLFLOW_ENABLED:
        return False
    parsed = urlparse(uri)
    try:
        with socket.create_connection((parsed.hostname, parsed.port), timeout=timeout):
            return True
    except OSError:
        return False


def make_logger():
    if not _mlflow_reachable(MLFLOW_TRACKING_URI):
        print(f"WARNING: MLflow недоступен ({MLFLOW_TRACKING_URI}) — без логгера")
        return None
    return MLFlowLogger(
        tracking_uri=MLFLOW_TRACKING_URI,
        experiment_name=MLFLOW_EXPERIMENT_NAME,
        save_dir=str(ROOT / "mlruns"),
    )


logger = make_logger()

In [ ]:
import lightning as L

trainer = L.Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    callbacks=[metrics_cb, pretrain_cb],
    logger=logger,
    precision="16-mixed",
    benchmark=True,
)
trainer.fit(dino, train_loader, val_dino_loader)

if logger is not None:
    print(f"MLflow run: {MLFLOW_TRACKING_URI}/#/experiments/{logger.experiment_id}/runs/{logger.run_id}")
print("Best pretrain ckpt:", pretrain_cb.best_model_path)

## 4. Фичи для оценки (teacher backbone)

In [ ]:
@torch.no_grad()
def embed_all(model, loader):
    model.eval().to(DEVICE)
    feats, labels = [], []
    for x, y in loader:
        feats.append(model.embed(x.to(DEVICE)).cpu())
        labels.append(y)
    return torch.cat(feats, 0), torch.cat(labels, 0)


tr_feats, tr_labels = embed_all(dino, train_plain_loader)
te_feats, te_labels = embed_all(dino, val_loader)
tr_feats.shape, te_feats.shape

## 5. k-NN

In [ ]:
import torch.nn.functional as F


@torch.no_grad()
def knn_predict(tr_feats, tr_labels, te_feats, k=20, temperature=0.07):
    sim = te_feats.to(DEVICE) @ tr_feats.to(DEVICE).t()
    vals, idx = sim.topk(k, dim=1)
    labels = tr_labels.to(DEVICE)[idx]
    weights = (vals / temperature).exp()

    votes = torch.zeros(te_feats.shape[0], 10, device=DEVICE)
    votes.scatter_add_(1, labels, weights)
    return votes.argmax(1)


preds = knn_predict(tr_feats, tr_labels, te_feats, k=20)
knn_acc = (preds.cpu() == te_labels).float().mean().item()
print(f"k-NN top-1 (k=20): {knn_acc:.4f}")

## 6. Linear probe

In [ ]:
import torchmetrics
import lightning as L
import torch.nn as nn
from lightning.pytorch.callbacks import EarlyStopping


class FeatureDataset(torch.utils.data.Dataset):
    # Обучает линейный классификатор поверх замороженных teacher-фич.

    def __init__(self, feats, labels):
        self.feats = feats
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.feats[i], self.labels[i]


class LinearProbe(L.LightningModule):
    def __init__(self, in_dim, num_classes=10, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.head = nn.Linear(in_dim, num_classes)
        self.acc = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)

    def forward(self, x):
        return self.head(x)

    def training_step(self, batch, i):
        x, y = batch
        loss = F.cross_entropy(self(x), y)
        self.log("train/loss", loss)
        return loss

    def validation_step(self, batch, i):
        x, y = batch
        loss = F.cross_entropy(self(x), y)
        self.log("val/loss", loss)
        self.acc(self(x), y)

    def on_validation_epoch_end(self):
        self.log("val/acc", self.acc.compute())
        self.acc.reset()

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=0.05)

In [ ]:
probe_ds = FeatureDataset(tr_feats, tr_labels)
probe_val = FeatureDataset(te_feats, te_labels)
probe_loader = torch.utils.data.DataLoader(probe_ds, batch_size=512, shuffle=True)
probe_val_loader = torch.utils.data.DataLoader(probe_val, batch_size=512)

probe = LinearProbe(in_dim=tr_feats.shape[1], num_classes=10, lr=1e-3)
probe_trainer = L.Trainer(
    max_epochs=20, accelerator="auto", log_every_n_steps=20,
    logger=logger, precision="16-mixed", benchmark=True,
    callbacks=[EarlyStopping(monitor="val/acc", mode="max", patience=5, verbose=True)],
)
probe_trainer.fit(probe, probe_loader, probe_val_loader)
print(f"Linear probe val acc: {probe_trainer.callback_metrics['val/acc']:.4f}")

## 7. Графики

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

m = metrics_cb.metrics
epochs = range(1, len(m["val/proto_acc"]) + 1)

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(epochs, m["train/loss_epoch"], label="train")
ax.plot(epochs, m["val/proto_acc"], label="val proto-acc")
ax.set_title("DINO loss / proto-acc")
ax.set_xlabel("epoch")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()